# Análisis de coherencia de outliers (CRIM, ZN, B)

La regla de IQR sola marca demasiados "outliers" en variables muy asimétricas (CRIM, ZN, B) que en realidad son la cola larga de una distribución real, no errores. Este análisis agrega un segundo filtro: para cada outlier, chequea si su relación con el target (MEDV) y con otras variables correlacionadas por dominio va en la dirección que el propio EDA (matriz de correlación) predice. Si un outlier rompe esa relación en 2 o más variables al mismo tiempo, se marca como **incoherente** y queda como candidato a revisión manual (no se elimina automáticamente).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Datos

Igual que en el resto del pipeline: nunca se imputa MEDV, se dropean sus nulos primero, antes de analizar cualquier otra cosa.

In [ ]:
df = pd.read_csv('house-prices-tp.csv')
df_limpio = df.dropna(subset=['MEDV']).copy()

## Función: detectar outliers por IQR

Marca los outliers de una columna con la regla clásica (Q1 - 1.5·IQR, Q3 + 1.5·IQR) y guarda de qué lado cae cada uno ('alto' o 'bajo'), que se usa después para juzgar coherencia.

In [ ]:
def detectar_outliers_iqr(df, columna):
    serie = df[columna].dropna()
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_outliers = df[(df[columna] < lim_inf) | (df[columna] > lim_sup)].copy()
    df_outliers['lado'] = np.where(df_outliers[columna] > lim_sup, 'alto', 'bajo')
    return df_outliers, lim_inf, lim_sup

## Función: chequear coherencia

Para los outliers de una columna, evalúa si el valor extremo es coherente con la relación esperada respecto al target y a otras variables correlacionadas por dominio (según la matriz de correlación del EDA).

- `direccion`: +1 si la columna correlaciona POSITIVO con el target, -1 si correlaciona NEGATIVO (esto sale de la matriz de correlación, no es un supuesto arbitrario).
- `relaciones`: dict opcional `{nombre_var: signo_esperado}` con otras variables relacionadas por dominio (ej: `{'LSTAT': 1, 'RM': -1}` para CRIM).

Cada outlier se compara contra la **mediana global** de cada variable (no la mediana de los outliers), y se cuenta cuántas relaciones esperadas rompe. Las filas con NaN en la variable comparada no se cuentan como incoherentes: no hay evidencia suficiente para juzgarlas.

In [ ]:
def chequear_coherencia(df, columna, direccion, relaciones=None, target='MEDV'):
    df_outliers, lim_inf, lim_sup = detectar_outliers_iqr(df, columna)
    signo_lado = np.where(df_outliers['lado'] == 'alto', 1, -1)

    def marca_rompe(var, signo_esperado):
        mediana_var = df[var].median(skipna=True)
        diff = df_outliers[var] - mediana_var
        signo_fila = np.sign(diff)
        signo_esperado_fila = signo_lado * signo_esperado
        valido = diff.notna() & (signo_fila != 0)
        return valido & (np.sign(signo_esperado_fila) != signo_fila)

    puntaje = marca_rompe(target, direccion).astype(int)
    df_outliers[f'rompe_{target}'] = marca_rompe(target, direccion)

    relaciones = relaciones or {}
    for var, signo_var in relaciones.items():
        rompe = marca_rompe(var, signo_var)
        df_outliers[f'rompe_{var}'] = rompe
        puntaje = puntaje + rompe.astype(int)

    df_outliers['puntaje_incoherencia'] = puntaje
    df_incoherentes = (
        df_outliers[df_outliers['puntaje_incoherencia'] >= 2]
        .sort_values('puntaje_incoherencia', ascending=False)
    )

    print(f"--- {columna} ---")
    print(f"  límites IQR: [{lim_inf:.2f}, {lim_sup:.2f}]  -> {len(df_outliers)} outliers totales")
    print(f"  incoherentes (rompen >=2 relaciones esperadas): {len(df_incoherentes)}")

    return df_outliers, df_incoherentes

## Función: graficar coherencia

Extiende el scatterplot "Impacto de X en el valor de la propiedad" ya existente, distinguiendo tres grupos: datos normales, outliers coherentes (se conservan) y outliers incoherentes (candidatos a revisión manual).

In [ ]:
def graficar_coherencia(df, columna, df_outliers, df_incoherentes, target='MEDV'):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x=columna, y=target, color='lightgrey', alpha=0.6,
                     label='Datos normales')

    coherentes = df_outliers.drop(df_incoherentes.index)
    sns.scatterplot(data=coherentes, x=columna, y=target, color='orange', edgecolor='black',
                     label='Outlier coherente (conservar)')

    sns.scatterplot(data=df_incoherentes, x=columna, y=target, color='red', edgecolor='black',
                     s=90, label='Outlier incoherente (revisar)')

    plt.title(f'Coherencia de outliers de {columna} respecto a {target}')
    plt.xlabel(columna)
    plt.ylabel(target)
    plt.legend()
    plt.show()

## Configuración: signos esperados por variable

Estos signos salen de la propia matriz de correlación del EDA (Sección "¿Cuál es la correlación entre las variables?"), no son un supuesto arbitrario:

- **CRIM** correlaciona negativo con MEDV, negativo con RM, positivo con LSTAT.
- **ZN** correlaciona positivo con MEDV, negativo con INDUS, negativo con LSTAT.
- **B** correlaciona positivo con MEDV, negativo con LSTAT.

In [ ]:
config = {
    'CRIM': dict(direccion=-1, relaciones={'RM': -1, 'LSTAT': 1}),
    'ZN':   dict(direccion=+1, relaciones={'INDUS': -1, 'LSTAT': -1}),
    'B':    dict(direccion=+1, relaciones={'LSTAT': -1}),
}

## Ejecutar el análisis para CRIM, ZN y B

In [ ]:
resultados = {}
for col, cfg in config.items():
    outliers, incoherentes = chequear_coherencia(df_limpio, col, **cfg)
    resultados[col] = (outliers, incoherentes)
    graficar_coherencia(df_limpio, col, outliers, incoherentes)

## Triangulación entre variables

Una fila que rompe el patrón esperado en **más de una** variable independiente es un candidato mucho más fuerte a revisión que una fila que solo lo rompe en una. No es automático que se elimine: es el punto de partida para decidir con criterio (diagnóstico de influencia, regresión robusta, o exclusión justificada y comparando métricas con/sin esas filas).

In [ ]:
indices = [set(incoh.index) for _, incoh in resultados.values()]
interseccion = set.union(*[
    indices[i] & indices[j]
    for i in range(len(indices)) for j in range(i + 1, len(indices))
])

print("--- Triangulación: filas incoherentes en MÁS DE UNA variable ---")
print(f"Filas candidatas a revisión prioritaria: {sorted(interseccion)}")

cols_mostrar = ['CRIM', 'ZN', 'B', 'RM', 'LSTAT', 'INDUS', 'MEDV']
df_limpio.loc[sorted(interseccion), cols_mostrar]